# Deep Agents and Sub-Agents - End to End (Groq-Working Edition)

### LangChain + LangGraph + Groq - with a **working alternative** for every example

> **Why this edition?** The Groq models (Llama / gpt-oss / qwen) do not support **native tool-calling** (`bind_tools`) well in this system. Sometimes they put the tool call into plain text, and Groq throws a `tool_use_failed (400)` error. So in this notebook **every concept** is shown in two ways:
>
> - **STANDARD way** (native tool-calling / `deepagents`) - this is the industry standard, but it can fail on Groq. Kept here as a reference so you understand it.
> - **GROQ-WORKING alternative** - does **not** use native tool-calling, so it runs cleanly on every Groq model. This is the one you should run and learn.

| Part | Concept | What is inside |
|------|---------|---------|
| 0 | Setup | Connect to Groq |
| 1 | **Foundation** | why it fails + build the Groq-working agent engine (the base for everything) |
| 2 | ReAct agent | standard vs Groq-working |
| 3 | Deep Agent 4 pillars | all Groq-working (planning / files / sub-agents) |
| 4 | `deepagents` package | standard (reference) + manual Groq-working equivalent |
| 5 | Research assistant | Groq-working |
| 6 | RAG support bot | Groq-working |
| 7 | Multi-step automation | Groq-working pipeline |
| 8 | Coding agent | Groq-working pipeline |
| 9 | Best practices + exercises | - |

---
### What is a Deep Agent?
A normal agent = **LLM + tools + loop** (ReAct). A **deep agent** = that agent plus 4 pillars: **(1) Planning** (todo), **(2) File system** (save notes), **(3) Sub-agents** (delegate to a specialist), **(4) Detailed prompt**. We will build all 4 in a Groq-working way.

## Part 0 - Setup

In [ ]:
%pip install -q -U langgraph langchain langchain-groq langchain-community \
    duckduckgo-search faiss-cpu langchain-huggingface sentence-transformers
print("install done")

In [1]:
import os
from getpass import getpass
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Groq API key: ")
print("key set")

key set


In [2]:
from langchain_groq import ChatGroq

# Approach matters more than the model. The working pattern in this notebook runs on ALL Groq models.
MODEL = "llama-3.3-70b-versatile"   # alt: "openai/gpt-oss-20b", "qwen/qwen3-32b"
llm = ChatGroq(model=MODEL, temperature=0, max_retries=3)

print(llm.invoke("In one line: what is the difference between an agent and a chatbot?").content)

c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The primary difference between an agent and a chatbot is that an agent is typically a human customer support representative, while a chatbot is an automated computer program designed to simulate human-like conversations and answer questions.


## Part 1 - Foundation: why Groq fails + the working engine

### The problem (see it yourself)
With native tool-calling, the Groq model sometimes puts the tool call into **plain text**:
```
<function=duckduckgo_search{"query":"..."}</function>
```
Groq cannot parse this, so you get `BadRequestError: tool_use_failed`. Below we try it with one tool (wrapped in try/except so it does not crash):

In [3]:
# STANDARD WAY (native tool calling) - can fail on Groq
from langchain_core.tools import tool

@tool
def multiply(a: float, b: float) -> float:
    '''Multiplies two numbers.'''
    return a * b

try:
    from langgraph.prebuilt import create_react_agent
    std_agent = create_react_agent(llm, [multiply])
    r = std_agent.invoke({"messages": [("user", "What is 23478 * 99211?")]})
    print("it worked this time:", r["messages"][-1].content)
except Exception as e:
    print("Groq tool_use_failed (this is your problem):\n", str(e)[:300])
    print("\nThe GROQ-WORKING engine below solves this.")

C:\Users\emon1\AppData\Local\Temp\ipykernel_2392\2191284694.py:11: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  std_agent = create_react_agent(llm, [multiply])


it worked this time: The result of 23478 * 99211 is 2329275858.


### The solution: a JSON-based agent engine
The idea: do **not** call `bind_tools` on the LLM. Instead, ask for **a JSON object** in plain text. We parse that JSON in Python and run the tool. If we never pass any tool to the API, the `tool_use_failed` error will **never** happen - and this works on **any** model.

This is our **core engine** - the whole notebook stands on this `run_agent`.

In [4]:
import json, re
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

def extract_json(text: str):
    '''Pulls the first valid JSON object out of text (handles ```fences / leading text).'''
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(text[start:i+1])
                except json.JSONDecodeError:
                    return None
    return None

def _tool_line(t):
    args = ", ".join(t.args.keys()) if getattr(t, "args", None) else ""
    return f"- {t.name}({args}): {t.description}"

print("extract_json + helper ready")
print(extract_json('blah ```json\n{"action":"final","answer":"hi"}\n```'))

extract_json + helper ready
{'action': 'final', 'answer': 'hi'}


In [5]:
def run_agent(llm, system_prompt, tools, user_input, max_steps=15, verbose=True):
    '''Groq-working ReAct loop. Does NOT use bind_tools. Runs on any model.'''
    tool_map = {t.name: t for t in tools}
    tool_block = "\n".join(_tool_line(t) for t in tools) if tools else "(no tools)"

    protocol = (
        "On every turn give ONLY one JSON object - no other text/markdown/```.\n"
        "Tool call:  {\"action\":\"tool\", \"tool\":\"<name>\", \"args\":{ ... }}\n"
        "Final:      {\"action\":\"final\", \"answer\":\"<full answer>\"}\n"
        "Rule: NOT a single character of text before or after the JSON. ONE action per turn.\n"
        "You will get the tool result as 'Observation:', then decide the next step."
    )
    sys = f"{system_prompt}\n\nAvailable tools:\n{tool_block}\n\n{protocol}"
    messages = [SystemMessage(content=sys), HumanMessage(content=user_input)]

    for step in range(max_steps):
        raw = llm.invoke(messages).content
        data = extract_json(raw)
        if data is None:
            messages.append(AIMessage(content=raw))
            messages.append(HumanMessage(content="That was not valid JSON. Give ONLY one JSON object."))
            continue
        if data.get("action") == "final":
            if verbose: print(f"  step {step}: FINAL")
            return data.get("answer", "")
        if data.get("action") == "tool":
            name, args = data.get("tool"), data.get("args", {})
            if name not in tool_map:
                obs = f"Error: tool '{name}' does not exist. Available: {list(tool_map)}"
            else:
                try:
                    obs = str(tool_map[name].invoke(args))
                except Exception as e:
                    obs = f"Tool '{name}' error: {e}"
            if verbose: print(f"  step {step}: {name}({args}) -> {str(obs)[:70]}")
            messages.append(AIMessage(content=raw))
            messages.append(HumanMessage(content=f"Observation: {obs}"))
        else:
            messages.append(HumanMessage(content="action must be 'tool' or 'final'."))
    return "max steps reached."

print("run_agent ready - this is the engine of the notebook")

run_agent ready - this is the engine of the notebook


In [6]:
# GROQ-WORKING DEMO - same model, but this time there is NO tool_use_failed
@tool
def add(a: float, b: float) -> float:
    '''Adds two numbers.'''
    return a + b

ans = run_agent(llm, "You are a careful math assistant. Do calculations using the tools.",
                [add, multiply],
                "First add 23478 and 100, then multiply the result by 99211.")
print("\nFINAL:", ans)

  step 0: add({'a': 23478, 'b': 100}) -> 23578.0


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kab8kywxe1ft6ma6z8g2n2wg` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 100000, Requested 222. Please try again in 3m11.808s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

**Same Groq model, no error.** That is because we did not pass any tool to the API - the LLM gives JSON and Python parses it. We will use this pattern everywhere from now on.

## Part 2 - ReAct Agent: standard vs Groq-working

**ReAct loop:** Reason -> Act (tool) -> Observe -> repeat -> final answer.

In [ ]:
# STANDARD WAY (reference) - robust on Anthropic/OpenAI, flaky on Groq
# from langgraph.prebuilt import create_react_agent
# agent = create_react_agent(llm, [add, multiply])
# agent.invoke({"messages":[("user","...")]})   # <- tool_use_failed risk on Groq

# GROQ-WORKING - same job, using run_agent
@tool
def word_count(text: str) -> int:
    '''Counts how many words are in the text.'''
    return len(text.split())

print(run_agent(llm, "You are a helpful assistant.", [add, multiply, word_count],
    "How many words are in 'deep agents are powerful'? And what is 888*444?"))

## Part 3 - The 4 Pillars of a Deep Agent (all Groq-working)

Now we add the 4 pillars on top of `run_agent` to build a full deep agent - all running on Groq.

### Pillar 1 - Planning (todo)  &  Pillar 2 - File system
Just give the tools to `run_agent` (no native calling needed). For the file system we share a dict (factory pattern):

In [ ]:
from langchain_core.tools import tool, StructuredTool

# --- Pillar 2: file system tools, on top of a shared dict ---
def make_file_tools(store: dict):
    @tool
    def write_file(filename: str, content: str) -> str:
        '''Writes or overwrites content in a file.'''
        store[filename] = content
        return f"saved {filename} ({len(content)} chars)"
    @tool
    def read_file(filename: str) -> str:
        '''Reads the content of a file.'''
        return store.get(filename, f"'{filename}' does not exist")
    @tool
    def list_files() -> str:
        '''Gives the names of all files.'''
        return ", ".join(store) or "(empty)"
    return [write_file, read_file, list_files]

# --- Pillar 1: planning tool (todo list in a dict) ---
def make_planning_tools(store: dict):
    @tool
    def write_todos(todos: list) -> str:
        '''Write the work plan as a todo list (list of strings).'''
        store["todos"] = [{"task": t, "done": False} for t in todos]
        return "Plan:\n" + "\n".join(f"  [ ] {t}" for t in todos)
    @tool
    def complete_todo(index: int) -> str:
        '''Mark the index-th todo (0-based) as done.'''
        ts = store.get("todos", [])
        if 0 <= index < len(ts):
            ts[index]["done"] = True
        return "\n".join(f"  [{'x' if t['done'] else ' '}] {t['task']}" for t in ts)
    return [write_todos, complete_todo]

print("planning + file tools factory ready")

### Pillar 3 - Sub-Agents (Groq-working)
**Pattern: agent-as-a-tool.** A sub-agent is itself a `run_agent` wrapped as a tool. When the main agent calls that tool, a full agent runs inside it (context isolation + expertise). It is all JSON-based, so it runs on Groq.

In [ ]:
def make_subagent_tool(name, description, llm, system_prompt, tools=None, max_steps=10):
    '''Turns a sub-agent into a tool for the main agent.'''
    sub_tools = tools or []
    def _run(task: str) -> str:
        return run_agent(llm, system_prompt, sub_tools, task, max_steps=max_steps, verbose=False)
    return StructuredTool.from_function(
        func=_run, name=name, description=description,
    )

# example sub-agents (use DuckDuckGo when search is needed)
from langchain_community.tools import DuckDuckGoSearchRun
search = DuckDuckGoSearchRun()

researcher_tool = make_subagent_tool(
    "research_subagent",
    "Researches a topic on the web and returns factual notes. Put the topic in 'task'.",
    llm,
    "You are a research specialist. Use the search tool to find 5-6 factual points + sources.",
    tools=[search],
)
writer_tool = make_subagent_tool(
    "writer_subagent",
    "Writes polished content from notes + instructions. Put notes+instructions in 'task'.",
    llm,
    "You are a professional writer. Write clear content from the given notes. Do not invent new facts.",
    tools=[],
)
print("2 sub-agent tools ready (researcher, writer)")

### Pillar 4 - Detailed prompt -> assemble the full Deep Agent (Groq-working)

In [ ]:
STATE = {}                       # shared file + todo store
file_tools = make_file_tools(STATE)
plan_tools = make_planning_tools(STATE)

DEEP_PROMPT = (
    "You are a deep agent that handles complex multi-step work.\n"
    "1. PLAN: for a big task, first use write_todos to make a plan.\n"
    "2. DELEGATE: need facts/search -> research_subagent; need writing -> writer_subagent.\n"
    "3. SAVE: save important notes/drafts with write_file, read them back later with read_file.\n"
    "4. TRACK: when a step is done, call complete_todo.\n"
    "5. FINISH: when everything is done, give the user the full final answer.\n"
    "Do not invent facts from your own head - use research_subagent."
)
deep_tools = plan_tools + file_tools + [researcher_tool, writer_tool]

def deep_agent(query, verbose=True):
    STATE.clear()
    return run_agent(llm, DEEP_PROMPT, deep_tools, query, max_steps=25, verbose=verbose)

print("Groq-working Deep Agent ready (4 pillars)")

In [ ]:
out = deep_agent("Make a ~120 word report on 'Electric vehicles in 2025'. "
                 "First research, then write, and manage it with a plan + files.")
print("\nFINAL REPORT:\n", out)
print("\nFiles:", [k for k in STATE if k != "todos"])
print("Todos:", STATE.get("todos"))

## Part 4 - The `deepagents` Package: standard reference + Groq-working equivalent

LangChain's **`deepagents`** package gives the 4 pillars built in. **But** internally it uses native tool-calling, so on Groq it **often gives `tool_use_failed`**. Your manual deep agent from Part 3 is the equivalent for Groq.

In [ ]:
# STANDARD WAY (reference) - great on Anthropic/OpenAI, unreliable on Groq
# %pip install deepagents
# from deepagents import create_deep_agent
# agent = create_deep_agent(
#     tools=[search],
#     instructions="You are a research assistant...",
#     subagents=[{"name":"researcher","description":"...","prompt":"...","tools":[search]}],
#     model=llm,          # <- with Groq the chance of tool_use_failed is high
# )
# agent.invoke({"messages":[("user","...")]})

print("Use the deepagents package with Anthropic/OpenAI models.")
print("   For Groq -> the manual deep_agent() from Part 3 = same concept, working.")

In [ ]:
# GROQ-WORKING equivalent of deepagents: a builder helper
def create_groq_deep_agent(instructions, tools=None, subagents=None, max_steps=25):
    '''A Groq-working version of deepagents.create_deep_agent (JSON engine + 4 pillars).'''
    state = {}
    base = make_planning_tools(state) + make_file_tools(state)
    sub_tools = [make_subagent_tool(s["name"], s["description"], llm,
                                    s["prompt"], s.get("tools", []))
                 for s in (subagents or [])]
    all_tools = base + (tools or []) + sub_tools
    full_prompt = instructions + (
        "\n\nYou can use planning (write_todos/complete_todo), file (write_file/read_file/list_files) "
        "and sub-agent tools. Plan the big task, save to files, "
        "and delegate to sub-agents."
    )
    def invoke(query, verbose=True):
        state.clear()
        ans = run_agent(llm, full_prompt, all_tools, query, max_steps=max_steps, verbose=verbose)
        return {"answer": ans, "files": {k: v for k, v in state.items() if k != "todos"},
                "todos": state.get("todos", [])}
    return invoke

# demo: researcher + critic, deepagents-style API but Groq-working
agent = create_groq_deep_agent(
    instructions="You are a report-writing orchestrator. Research -> draft -> review with critic -> improve.",
    tools=[search],
    subagents=[
        {"name": "researcher", "description": "researches a topic on the web and returns notes ('task' = topic)",
         "prompt": "You are a research specialist. Use search to give factual notes + sources.", "tools": [search]},
        {"name": "critic", "description": "reviews a draft and finds weak points ('task' = draft)",
         "prompt": "You are a strict editor. Give the draft's gaps/improvements as bullets.", "tools": []},
    ],
)
res = agent("Write a ~100 word brief on 'Remote work productivity': research -> draft -> critic -> final.")
print("\n", res["answer"])
print("\nfiles:", list(res["files"]))

## Part 5 - Research Assistant (Groq-working)
Orchestrator deep agent + research sub-agent -> a multi-section report. Built fully with the Groq builder from Part 4.

In [ ]:
research_assistant = create_groq_deep_agent(
    instructions=(
        "You are a senior research analyst. When you get a topic:\n"
        "1. Use write_todos to split it into 3 sub-topics.\n"
        "2. Give each sub-topic to 'researcher', save the notes in separate files.\n"
        "3. Read all the notes and write a structured report: Title, Summary, sections, Key Takeaways.\n"
        "4. Give the final report to the user. Use only researched facts."
    ),
    tools=[search],
    subagents=[{"name": "researcher",
                "description": "researches a sub-topic on the web and returns notes ('task' = sub-topic)",
                "prompt": "You are an expert researcher. Use search to give 5-6 factual points + sources. No speculation.",
                "tools": [search]}],
    max_steps=40,
)
out = research_assistant("Make a research report: 'Impact of AI agents on software jobs in 2025'")
print(out["answer"][:1800])
print("\nfiles:", list(out["files"]))

## Part 6 - Customer Support RAG Bot (Groq-working)

RAG = retrieve the relevant doc -> then answer. **Retrieval does not need tool-calling** - you can retrieve in Python and put it in the prompt, or give `run_agent` a retrieve tool (JSON-based, Groq-working). Embeddings are local (Groq does not provide embeddings).

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

KB = [
 "Refund: full refund within 30 days. After 30 days only store credit. Digital products non-refundable once downloaded.",
 "Shipping: standard 5-7 days, free over $50. Express (2 days) $15. Ships to US, Canada, EU.",
 "Account: reset password via Settings > Security. Account deletion permanent, done within 48h.",
 "Subscription: Basic $9/mo, Pro $29/mo, Enterprise custom. Cancel anytime, access till cycle end, no partial refund.",
 "Support: live chat Mon-Fri 9am-6pm EST. After hours email support@company.com, reply within 24h.",
]
emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vs = FAISS.from_documents([Document(page_content=d) for d in KB], emb)
retriever = vs.as_retriever(search_kwargs={"k": 2})
print("vector store ready")

In [ ]:
# STANDARD WAY: wrap the retriever as a native tool with create_react_agent -> flaky on Groq
# GROQ-WORKING: retrieve tool + run_agent (JSON engine)
@tool
def search_knowledge_base(query: str) -> str:
    '''Search the company policy/FAQ knowledge base for relevant info.'''
    hits = retriever.invoke(query)
    return "\n".join(f"[{i+1}] {h.page_content}" for i, h in enumerate(hits))

@tool
def escalate_to_human(reason: str) -> str:
    '''If it is not in the KB or is sensitive (legal/complaint/dispute), escalate to a human agent.'''
    return f"Ticket created, a human agent will contact you. (reason: {reason})"

SUPPORT_PROMPT = (
    "You are a polite customer support agent.\n"
    "1. ALWAYS call search_knowledge_base before answering.\n"
    "2. Answer ONLY with the retrieved info, do not make things up.\n"
    "3. If it is not in the KB, or it is a complaint/legal/dispute, call escalate_to_human.\n"
    "4. Tone: friendly, concise, professional."
)
def support(q):
    a = run_agent(llm, SUPPORT_PROMPT, [search_knowledge_base, escalate_to_human], q,
                  max_steps=8, verbose=False)
    print(f"User: {q}\nBot: {a}\n" + "-"*55)

support("I bought something 10 days ago, can I get a full refund?")
support("How much is express shipping and how long?")
support("I want to sue you for a defective product.")     # -> escalate
support("Do you have an Android app?")                     # -> not in KB -> escalate

## Part 7 - Multi-Step Automation (Groq-working pipeline)

**Teaching point:** not every deep agent has to be an autonomous loop. In production, a **fixed pipeline + LLM sub-agents** is often more reliable and predictable. Here a sub-agent = a focused single-shot LLM call (`ask`) - 100% stable on Groq.

In [ ]:
import json, statistics
from langchain_core.messages import SystemMessage, HumanMessage

def ask(role_prompt, task):
    '''A focused single-shot sub-agent call (no loop -> fully reliable).'''
    return llm.invoke([SystemMessage(content=role_prompt), HumanMessage(content=task)]).content

SALES = [{"month":"Jan","rev":42000},{"month":"Feb","rev":38000},{"month":"Mar","rev":51000},
         {"month":"Apr","rev":47000},{"month":"May","rev":63000},{"month":"Jun","rev":58000}]

def automation_pipeline(request, verbose=True):
    files = {}
    # 1) analyst (deterministic compute + LLM summary)
    revs = [r["rev"] for r in SALES]
    stats = {"total": sum(revs), "avg": round(statistics.mean(revs)),
             "growth_%": round((revs[-1]-revs[0])/revs[0]*100, 1), "peak": max(SALES, key=lambda r:r["rev"])}
    if verbose: print("[analyst] stats:", stats)
    files["analysis.md"] = ask("You are a data analyst. Give clear findings (bullets) from the numbers.",
                               f"Request: {request}\nData: {SALES}\nStats: {stats}")
    # 2) insight-generator
    if verbose: print("[insighter] generating insights...")
    files["insights.md"] = ask("You are a business strategist. From the findings give 3 actionable insights + recommendations.",
                               files["analysis.md"])
    # 3) email-writer
    if verbose: print("[emailer] drafting email...")
    files["email.md"] = ask("You are a business writer. From the insights write a concise email for leadership: subject, greeting, bullets, recommendation, sign-off.",
                            files["insights.md"])
    return files

files = automation_pipeline("Analyze H1 sales, find insights, draft a leadership email.")
print("\nFINAL EMAIL:\n", files["email.md"])
print("\nfiles:", list(files))

## Part 8 - Coding Agent (Groq-working: plan -> code -> test -> fix -> review)

A pipeline + sub-agents. The generated code is verified with real assert-tests; if it fails, the agent **debugs itself** and retries. No native tool-calling -> stable on Groq.

In [ ]:
import io, contextlib, traceback, re

def extract_code(text):
    m = re.search(r"```(?:python)?\s*(.*?)```", text, re.DOTALL)
    return (m.group(1) if m else text).strip()

def run_tests(code, tests):
    '''WARNING: runs LLM code LOCALLY with exec - only run in a trusted env.'''
    buf, ns = io.StringIO(), {}
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, ns); exec(tests, ns)
        return True, "all tests pass"
    except Exception:
        return False, traceback.format_exc().splitlines()[-1]

def coding_agent(spec, tests, max_fix=3, verbose=True):
    plan = ask("You are a senior engineer. Break the spec into 3-5 implementation steps. Plan, not code.", spec)
    if verbose: print("plan ready")
    err, code = "", ""
    for i in range(max_fix + 1):
        task = f"Spec:\n{spec}\nPlan:\n{plan}\nTests (MUST pass):\n{tests}"
        if err: task += f"\n\nThe previous code gave this error, FIX it:\n{err}\nPrevious code:\n{code}"
        code = extract_code(ask("You are an expert Python dev. Give ONLY one ```python function, with a docstring.", task))
        ok, out = run_tests(code, tests)
        if verbose: print(f"attempt {i+1}: {out}")
        if ok: break
        err = out
    review = ask("You are a strict reviewer. Give the code's edge cases/bugs/improvements as bullets.", code)
    return {"code": code, "passed": ok, "plan": plan, "review": review}

spec = ("Function merge_intervals(intervals): list of [start,end] int intervals (unsorted, can overlap). "
        "Merge overlapping ones and return a non-overlapping sorted list. e.g. [[1,3],[2,6],[8,10]] -> [[1,6],[8,10]]")
tests = ("assert merge_intervals([[1,3],[2,6],[8,10],[15,18]])==[[1,6],[8,10],[15,18]]\n"
         "assert merge_intervals([])==[]\n"
         "assert merge_intervals([[1,4],[0,4]])==[[0,4]]\n"
         "assert merge_intervals([[1,4],[4,5]])==[[1,5]]")
res = coding_agent(spec, tests)
print(f"\nPassed: {res['passed']}\n\nCODE:\n{res['code']}")

In [ ]:
# does the generated function really work?
exec(res["code"], globals())
print(merge_intervals([[1,3],[2,6],[8,10],[15,18]]))
print(merge_intervals([[1,4],[0,2],[3,5]]))
print("\nREVIEW:\n", res["review"][:500])

## Part 9 - Best Practices + Exercises

### Production tips
1. **For Groq + agents, use the JSON/pipeline pattern** (this notebook), not native `bind_tools`/`deepagents` - those give `tool_use_failed`.
2. **When to use a sub-agent?** When you need separate expertise/prompt/context-isolation. Not for a simple single-step task (overkill).
3. **Loop vs pipeline:** uncertain/open-ended work = `run_agent` loop; fixed workflow = deterministic pipeline (Part 7/8) - more reliable.
4. **Error handling:** use try/except in tools, give the LLM clear error feedback (self-debug, shown in Part 8).
5. **Set max_steps / max_retries** - runaway loop and cost control.
6. **Observability:** turn on LangSmith (`LANGCHAIN_TRACING_V2=true`) - debug by looking at the step trace.
7. **Validation:** retry if JSON parsing fails (built into run_agent). Validate critical output with Pydantic.
8. **Hybrid:** do reasoning/drafting on Groq (fast/cheap), do the critical tool-calling step on OpenAI/Anthropic (robust).

### Standard vs Groq-working - quick table
| Concept | Standard (Anthropic/OpenAI) | Groq-working (this notebook) |
|--------|---------------------------|---------------------------|
| Tool calling | `llm.bind_tools()` | `run_agent` (JSON parse) |
| ReAct agent | `create_react_agent` | `run_agent` |
| Deep agent | `deepagents.create_deep_agent` | `create_groq_deep_agent` |
| Sub-agent | `subagents=[...]` | `make_subagent_tool` |
| Workflow | autonomous agent | deterministic pipeline + `ask` |

### Exercises
1. **Easy:** In the Part 6 support bot, put 5-10 FAQs for your own product.
2. **Easy:** In the Part 5 research assistant, add a `tweet_writer` sub-agent.
3. **Medium:** In `run_agent`, return a `verbose` trace (all steps in a list).
4. **Medium:** In the Part 8 coding agent, add an `optimize` step (make working code faster).
5. **Hard:** Build a multi-agent "startup team": PM -> engineer -> QA sub-agents, going from a feature spec to plan+code+test.
6. **Hard:** In `run_agent`, validate tool args with Pydantic; if args are wrong, give the LLM fix-feedback.

---
**Done!** This edition has a **Groq-working** version of every concept, so you can learn from it and build real projects. Groq's native tool-calling is weak, but with this JSON/pipeline pattern a full deep-agent system runs on Groq.